# 正负混合比的合成注入

用 0.1.1 候选源码的三个内置模式，分别在 `delta = -1` 和 `+1`、`sigma/I = 0.3` 生成合成观测，再用同一个核心反演。偏振度的符号需由具体响应计算。偏振误差幅度取 `max(0.1075, 0.8 * abs(P))`，角分布比或 DCO 比的误差幅度取 `0.1 * abs(R)`。这里使用示例自带的探测器组和扫描范围。

This small same-core round trip demonstrates signed injections and reporting of all angle-selected 68% HDI components.

In [ ]:
from dataclasses import replace
from pprint import pprint

import numpy as np

from multipolepdm.contracts import Measurement
from multipolepdm.io.input import example_data, job_from_dict
from multipolepdm.jobs.workers import compute_job
from multipolepdm.models.registry import evaluate_response

records = []
input_width_ratio = 0.3
for mode in ('P_ADO', 'P_DCO', 'PDCO'):
    template = job_from_dict(example_data(mode))
    for input_delta in (-1.0, 1.0):
        response = evaluate_response(
            template,
            np.array([np.arctan(input_delta)]),
            np.array([input_width_ratio * template.reference_spin]),
        )
        polarization = float(response.polarization[0, 0])
        ratio = float(response.angular_ratio[0, 0])
        polarization_error = max(0.1075, 0.8 * abs(polarization))
        ratio_error = 0.1 * abs(ratio)
        job = replace(
            template,
            label=f'{mode}: synthetic delta={input_delta:+g}',
            polarization=Measurement(polarization, polarization_error, polarization_error),
            angular_ratio=Measurement(ratio, ratio_error, ratio_error),
        )
        result, summary = compute_job(job)
        tan_interval = summary.tan_angle_statistics.intervals[0]
        width_interval = summary.sigma_over_i.intervals[0]
        records.append({
            'mode': mode,
            'input_delta': input_delta,
            'input_sigma_over_I': input_width_ratio,
            'source_spin': result.response.reference_spin,
            'P': polarization,
            'ratio': ratio,
            'tan{q50[arctan(δ)]}': summary.tan_angle_statistics.tan_median,
            'tan{ETI68%[arctan(δ)]}': tan_interval.tan_eti,
            'tan{HDI68%[arctan(δ)]}': [
                (segment.bounds, segment.mass, segment.mode_ranges)
                for segment in tan_interval.tan_hdi_segments
            ],
            'input_in_tan_angle_HDI68': any(
                segment.bounds[0] <= input_delta <= segment.bounds[1]
                for segment in tan_interval.tan_hdi_segments
            ),
            'sigma_over_I_q50': summary.sigma_over_i.median,
            'sigma_over_I_ETI_68': width_interval.eti,
            'sigma_over_I_HDI_68': [
                (segment.bounds, segment.mass, segment.mode_ranges)
                for segment in width_interval.hdi_segments
            ],
            'warnings': result.warnings,
        })

for record in records:
    pprint(record, sort_dicts=False)

每个 HDI 条目依次是端点、该段的概率质量和段内峰位范围。区间较宽或分成多段时保留所有分支。`input_in_tan_angle_HDI68` 仅检查这里给定的合成观测，覆盖率需对指定采样模型重复生成含噪数据。独立正向模型和文献比较使用配套的验证脚本及几何来源，文献误差沿用原文口径。 这些区间是tan{HDI68%[arctan(δ)]}，不称δ PDF自身的HDI；各段mode_ranges也是角局部峰的正切表示。此Notebook只更新读取与显示命名，本轮不执行或改写历史计算。